# Phase 6 — Cross-Method Benchmarks & Analysis

**Goal:** assemble a single picture from all method-specific notebooks. This is the notebook that produces the headline tables for the thesis writeup.

**Inputs (Kaggle datasets):**
- `sqat-baseline`     — FP32 numbers from notebook 01
- `sqat-ptq`          — PTQ INT8 / INT4
- `sqat-standard-qat` — Standard QAT INT8 / INT4
- `sqat-scheduled-qat` — Scheduled QAT linear / cosine / step
- `sqat-lora-qat`     — LoRA-QAT INT8 / INT4
- `sqat-gguf` *(optional)* — GGUF file sizes

**Outputs:**
- `results/tables/primary_results.csv`
- `results/tables/schedule_comparison.csv`
- `results/plots/*.png`

This notebook is CPU-only — no GPU required.

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q -e ".[viz]"

In [ ]:
import json
import math
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

TABLES_DIR = Path(WORKING_DIR) / "results" / "tables"
PLOTS_DIR  = Path(WORKING_DIR) / "results" / "plots"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load all results

Each phase notebook writes a per-experiment `training_results.json` plus a method-level summary JSON. We glob both, merge into one DataFrame, then add derived columns.

In [ ]:
DATASET_ROOTS = [
    Path("/kaggle/input/sqat-baseline"),
    Path("/kaggle/input/sqat-ptq"),
    Path("/kaggle/input/sqat-standard-qat"),
    Path("/kaggle/input/sqat-scheduled-qat"),
    Path("/kaggle/input/sqat-lora-qat"),
    Path(WORKING_DIR),                # in case you re-ran phases here
]

# Approximate model sizes (GB) per bit-width. Real GGUF sizes can replace these
# if you mount sqat-gguf — see Section 6.
APPROX_SIZES = {32: 6.5, 16: 3.4, 8: 1.7, 4: 0.85}

def find_results():
    rows = []
    for root in DATASET_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob("training_results.json"):
            with path.open() as f:
                d = json.load(f)
            d["_source"] = str(path)
            rows.append(d)
    return rows

def load_baseline():
    for root in DATASET_ROOTS:
        p = root / "results" / "baseline" / "baseline_results.json"
        if p.exists():
            with p.open() as f:
                return json.load(f)
    return None

fp32 = load_baseline()
raw = find_results()
print(f"Loaded {len(raw)} experiment result files.")
if fp32:
    print(f"FP32 perplexity: {fp32.get('perplexity'):.4f}")

In [ ]:
def schedule_of(name: str) -> str:
    # scheduled_qat_cosine_int4 -> cosine ; standard_qat_int4 -> '' ; ptq_int4 -> ''
    parts = name.split("_")
    if parts[:2] == ["scheduled", "qat"] and len(parts) >= 4:
        return parts[2]
    return ""

rows = []
fp32_ppl = fp32["perplexity"] if fp32 else None

for d in raw:
    name = d.get("experiment_name", Path(d["_source"]).parent.name)
    bits = d.get("target_bits")
    method = d.get("method", name.split("_int")[0])
    ppl = d.get("perplexity")
    rows.append({
        "experiment_name": name,
        "method":          method,
        "schedule":        schedule_of(name),
        "bits":            bits,
        "perplexity":      ppl,
        "kl_divergence":   d.get("kl_divergence"),
        "final_loss":      d.get("final_loss"),
        "training_time_s": d.get("training_time_seconds"),
        "size_gb":         APPROX_SIZES.get(bits),
        "ppl_delta_pct":   ((ppl / fp32_ppl) - 1) * 100 if (ppl and fp32_ppl) else None,
    })

df = pd.DataFrame(rows).sort_values(["method", "bits", "schedule"]).reset_index(drop=True)
df.round(4)

## 3. Primary results table

The headline table from `SKILL.md`. Lower PPL and lower KLD = better.

In [ ]:
primary_cols = ["method", "schedule", "bits", "perplexity", "ppl_delta_pct",
                "kl_divergence", "size_gb", "training_time_s"]
primary = df[primary_cols].copy()

# Prepend the FP32 baseline row.
if fp32:
    primary = pd.concat([
        pd.DataFrame([{
            "method": "baseline", "schedule": "", "bits": 32,
            "perplexity": fp32["perplexity"], "ppl_delta_pct": 0.0,
            "kl_divergence": 0.0, "size_gb": 6.5, "training_time_s": 0,
        }]),
        primary,
    ], ignore_index=True)

primary = primary.round({"perplexity": 4, "ppl_delta_pct": 2,
                          "kl_divergence": 6, "size_gb": 2, "training_time_s": 0})
primary.to_csv(TABLES_DIR / "primary_results.csv", index=False)
primary

## 4. Schedule comparison (INT4)

The thesis-specific table. Which schedule wins, and by how much?

In [ ]:
sched_df = df[(df["method"] == "scheduled_qat") & (df["bits"] == 4)][
    ["schedule", "perplexity", "ppl_delta_pct", "kl_divergence", "final_loss", "training_time_s"]
].sort_values("kl_divergence").reset_index(drop=True)
sched_df = sched_df.round({"perplexity": 4, "ppl_delta_pct": 2,
                            "kl_divergence": 6, "final_loss": 4, "training_time_s": 0})
sched_df.to_csv(TABLES_DIR / "schedule_comparison.csv", index=False)
sched_df

## 5. Plots

In [ ]:
# Perplexity grouped by method × bits.
plot_df = df.dropna(subset=["perplexity"]).copy()
if not plot_df.empty:
    pivot = plot_df.pivot_table(
        index="bits", columns="method", values="perplexity", aggfunc="min",
    ).sort_index()
    fig = go.Figure()
    for col in pivot.columns:
        fig.add_trace(go.Bar(name=col, x=pivot.index.astype(str), y=pivot[col],
                             text=[f"{v:.2f}" if pd.notna(v) else "" for v in pivot[col]],
                             textposition="outside"))
    if fp32:
        fig.add_hline(y=fp32["perplexity"], line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32['perplexity']:.3f}")
    fig.update_layout(barmode="group", title="WikiText-103 perplexity by method × bits",
                      xaxis_title="bits", yaxis_title="perplexity (lower=better)",
                      height=480, margin=dict(t=80, b=40, l=60, r=20))
    fig.write_html(PLOTS_DIR / "ppl_vs_bits.html")
    fig.show()

In [ ]:
kld_df = df.dropna(subset=["kl_divergence"])
if not kld_df.empty:
    pivot = kld_df.pivot_table(
        index="method", columns="bits", values="kl_divergence", aggfunc="min",
    )
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=[f"INT{b}" for b in pivot.columns],
        y=pivot.index,
        colorscale="Viridis",
        text=[[f"{v:.4f}" if pd.notna(v) else "" for v in row] for row in pivot.values],
        texttemplate="%{text}",
        colorbar=dict(title="KLD"),
        hoverongaps=False,
    ))
    fig.update_layout(title="KL divergence vs FP32 (lower=better)",
                      xaxis_title="bits", yaxis_title="method",
                      height=420, margin=dict(t=80, b=40, l=120, r=20))
    fig.write_html(PLOTS_DIR / "kld_heatmap.html")
    fig.show()

In [ ]:
if not sched_df.empty:
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "INT4 perplexity by schedule", "INT4 KL divergence by schedule"))
    fig.add_trace(go.Bar(x=sched_df["schedule"], y=sched_df["perplexity"],
                         marker_color=["#4C72B0", "#DD8452", "#55A868"][: len(sched_df)],
                         text=[f"{v:.3f}" for v in sched_df["perplexity"]],
                         textposition="outside"), 1, 1)
    if fp32:
        fig.add_hline(y=fp32["perplexity"], line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32['perplexity']:.3f}", row=1, col=1)
    fig.add_trace(go.Bar(x=sched_df["schedule"], y=sched_df["kl_divergence"],
                         marker_color=["#4C72B0", "#DD8452", "#55A868"][: len(sched_df)],
                         text=[f"{v:.4f}" for v in sched_df["kl_divergence"]],
                         textposition="outside"), 1, 2)
    fig.update_xaxes(title_text="schedule")
    fig.update_yaxes(title_text="perplexity", row=1, col=1)
    fig.update_yaxes(title_text="KL divergence", row=1, col=2)
    fig.update_layout(showlegend=False, height=420,
                      margin=dict(t=80, b=40, l=60, r=20))
    fig.write_html(PLOTS_DIR / "schedule_comparison.html")
    fig.show()

In [ ]:
scatter = df.dropna(subset=["perplexity", "size_gb"]).copy()
if not scatter.empty:
    scatter["label"] = scatter.apply(
        lambda r: f"{r['schedule']} INT{int(r['bits'])}" if r["schedule"] else f"INT{int(r['bits'])}",
        axis=1,
    )
    fig = px.scatter(scatter, x="size_gb", y="perplexity", color="method",
                     symbol="method", size_max=14, hover_data=["label", "kl_divergence", "ppl_delta_pct"],
                     text="label",
                     title="Pareto view — quality vs model size")
    fig.update_traces(marker=dict(size=14), textposition="top center")
    if fp32:
        fig.add_trace(go.Scatter(x=[6.5], y=[fp32["perplexity"]], mode="markers+text",
                                 marker=dict(size=20, color="black", symbol="star"),
                                 text=["FP32"], textposition="top center", name="FP32"))
    fig.update_layout(xaxis_title="approx. model size (GB)",
                      yaxis_title="perplexity (lower=better)",
                      height=520, margin=dict(t=80, b=40, l=60, r=20))
    fig.write_html(PLOTS_DIR / "quality_vs_size.html")
    fig.show()

## 6. (Optional) Real GGUF sizes

If you mount the `sqat-gguf` dataset, this cell replaces the approximate sizes with measured `.gguf` file sizes.

In [ ]:
GGUF_ROOT = Path("/kaggle/input/sqat-gguf")
if GGUF_ROOT.exists():
    rows = []
    for p in GGUF_ROOT.rglob("*.gguf"):
        rows.append({"name": p.name, "size_gb": round(p.stat().st_size / 1e9, 3)})
    gguf_df = pd.DataFrame(rows).sort_values("size_gb")
    print(gguf_df.to_string(index=False))
    gguf_df.to_csv(TABLES_DIR / "gguf_sizes.csv", index=False)
else:
    print("sqat-gguf dataset not mounted — skipping. Approx sizes used in tables above.")

## 7. Conclusions cell

Auto-summarises the headline findings: best INT4 method, best schedule, and Scheduled-vs-Standard QAT improvement.

In [ ]:
lines = []

int4 = df[df["bits"] == 4].dropna(subset=["kl_divergence"])
if not int4.empty:
    best = int4.sort_values("kl_divergence").iloc[0]
    lines.append(f"Best INT4 by KLD: {best['experiment_name']}  "
                 f"(KLD={best['kl_divergence']:.6f}, PPL={best['perplexity']:.4f})")

if not sched_df.empty:
    winner = sched_df.iloc[0]
    lines.append(f"Best schedule (INT4): {winner['schedule']}  "
                 f"(KLD={winner['kl_divergence']:.6f})")

std_int4 = df[(df["method"] == "standard_qat") & (df["bits"] == 4)]
sch_int4 = df[(df["method"] == "scheduled_qat") & (df["bits"] == 4)]
if not std_int4.empty and not sch_int4.empty:
    std_kld = std_int4["kl_divergence"].min()
    sch_kld = sch_int4["kl_divergence"].min()
    if pd.notna(std_kld) and pd.notna(sch_kld) and std_kld > 0:
        improvement = (std_kld - sch_kld) / std_kld * 100
        lines.append(f"Scheduled vs Standard QAT @ INT4: "
                     f"KLD {std_kld:.6f} -> {sch_kld:.6f}  ({improvement:+.1f}%)")

for l in lines:
    print(l)

with (TABLES_DIR / "conclusions.txt").open("w") as f:
    f.write("\n".join(lines) + "\n")

## 8. Cross-method sample inference

Each method notebook (02–05) saves its own `<method>_inference.json` containing the same five-prompt completions. This cell merges all of them into one wide table so you can read across columns and *see* the quality differences side-by-side — the qualitative complement to the metrics above.

In [ ]:
from IPython.display import display, HTML

INFERENCE_FILES = [
    ("ptq",           "ptq_inference.json"),
    ("standard_qat",  "standard_qat_inference.json"),
    ("scheduled_qat", "scheduled_qat_inference.json"),
    ("lora_qat",      "lora_qat_inference.json"),
    ("gguf",          "gguf_inference.json"),
]

merged = None
for label, fname in INFERENCE_FILES:
    for root in DATASET_ROOTS:
        p = root / "results" / fname
        if p.exists():
            sub = pd.read_json(p)
            # FP32 column shows up in every per-method file; keep it from the first one only.
            if merged is None:
                merged = sub.copy()
            else:
                cols_to_add = [c for c in sub.columns if c not in merged.columns]
                merged = pd.concat([merged, sub[cols_to_add]], axis=1)
            print(f"  loaded {p}")
            break

if merged is not None and not merged.empty:
    merged.to_csv(TABLES_DIR / "cross_method_inference.csv", index=False)
    display(HTML(merged.to_html(index=False, escape=False).replace(
        "<table", '<table style="font-family:monospace; font-size:11px"')))
else:
    print("No inference JSONs found. Run notebooks 02-06 first and save outputs to Drive/Kaggle dataset.")

In [ ]:
!ls -lh {TABLES_DIR} {PLOTS_DIR}